In [77]:
%%configure -f
{"executorMemory": "12G", "executorCores": 12, "ttl": "12h", "heartbeatTimeoutInSecond": 43200, "numExecutors": 3}

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
11,None,pyspark,idle,,,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
11,None,pyspark,idle,,,None,✔


In [ ]:
gm12878_query_id = "a1fc46a9-93f8-424f-b41d-37bfd85d3b94"
h1esc_query_id = "f7bc6dac-6aa3-49e6-a2e5-c2ff27824c81"
hffc6_query_id = "f648f805-c3a9-4cf4-a108-94e6f5fa96c1"

In [78]:
import pyspark.sql.functions as F

results_gm12878_df = (
    spark
    .read
    .parquet(f"s3a://database/results/{gm12878_query_id}")
    .withColumn("cell_line", F.lit("GM12878"))
)

results_h1esc_df = (
    spark
    .read
    .parquet(f"s3a://database/results/{h1esc_query_id}")
    .withColumn("cell_line", F.lit("H1ESC"))
)

results_hffc6_df = (
    spark
    .read
    .parquet(f"s3a://database/results/{hffc6_query_id}")
    .withColumn("cell_line", F.lit("HFFC6"))
)

results = results_gm12878_df.union(results_h1esc_df).union(results_hffc6_df)

chromatin_states_df = (
    spark
    .read
    .parquet("s3a://database/chromatin_states")
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
active_gene_states = ['TssA', 'TssAFlnk', 'TxFlnk', 'Tx', 'TxWk', 'EnhG', 'EnhG1', 'EnhG2', 'Enh', 'EnhA1', 'EnhA2']
used_projects = ['whole_all_vs_all_gm12878_fix', 'whole_all_vs_all_h1esc_fix', 'whole_all_vs_all_hffc6_fix']

# CCD (contact domain) breakpoints — single genomic positions; two loci are in the SAME CCD iff
# NO breakpoint lies strictly between them. Upload data/ref/<CELL>/*.breakpoints.bed to these S3
# paths (set CCD_BREAKPOINTS = None to disable the same-CCD filter).
CCD_BREAKPOINTS = {
    "GM12878": "s3a://database/ccd_breakpoints/ccds_all_hg38_merged100k_GM12878.breakpoints.bed",
    "H1ESC":   "s3a://database/ccd_breakpoints/ccds_all_hg38_merged100k_H1ESC.breakpoints.bed",
    "HFFC6":   "s3a://database/ccd_breakpoints/ccds_all_hg38_merged100k_HFFC6.breakpoints.bed",
}

chromatin_states_df = (
    chromatin_states_df
    .where(F.col('name').isin(active_gene_states))
)

results = (
    results
    .where("avg_dist > 0 AND var_dist > 0")
    .where(F.col('project_id').isin(used_projects))
)

In [ ]:
results.createOrReplaceTempView("results")
chromatin_states_df.createOrReplaceTempView("chromatin_states")

# CCD breakpoints (single positions) per cell line -> view (cell_line, chrom, pos)
USE_CCD_FILTER = bool(CCD_BREAKPOINTS)
if USE_CCD_FILTER:
    breakpoints_df = None
    for cl, path in CCD_BREAKPOINTS.items():
        bdf = (spark.read.option("sep", "\t").schema("chrom string, start long, end long").csv(path)
               .select("chrom", F.col("start").alias("pos"))
               .withColumn("cell_line", F.lit(cl)))
        breakpoints_df = bdf if breakpoints_df is None else breakpoints_df.union(bdf)
    breakpoints_df.createOrReplaceTempView("breakpoints")
    print("CCD breakpoints loaded:", breakpoints_df.groupBy("cell_line").count().collect())

# same-CCD filter: gene TSS and enhancer centre lie in ONE contact domain (no breakpoint between)
_ccd_clause = ""
if USE_CCD_FILTER:
    _ccd_clause = """
AND r.gene_chr = r.enh_chr
AND NOT EXISTS (
    SELECT 1 FROM breakpoints bp
    WHERE bp.cell_line = r.cell_line AND bp.chrom = r.gene_chr
      AND bp.pos > least(CASE WHEN r.gene_strand = '+' THEN r.gene_start ELSE r.gene_end END,
                         (r.enh_start + r.enh_end) div 2)
      AND bp.pos < greatest(CASE WHEN r.gene_strand = '+' THEN r.gene_start ELSE r.gene_end END,
                            (r.enh_start + r.enh_end) div 2))"""

query = f"""
SELECT DISTINCT
    r.project_id,
    r.gene_id,
    r.enh_id,
    r.avg_dist,
    r.var_dist,
    r.cell_line
FROM results r
WHERE EXISTS (
    SELECT 1
    FROM chromatin_states cs_gene
    WHERE cs_gene.cell_line = r.cell_line
      AND cs_gene.chrom = r.gene_chr
      AND cs_gene.start <= r.gene_end
      AND cs_gene.end >= r.gene_start
)
AND EXISTS (
    SELECT 1
    FROM chromatin_states cs_enh
    WHERE cs_enh.cell_line = r.cell_line
      AND cs_enh.chrom = r.enh_chr
      AND cs_enh.start <= r.enh_end
      AND cs_enh.end >= r.enh_start
){_ccd_clause}
"""
results = spark.sql(query)

In [81]:
results_by_gene = (
    results
    .groupBy('project_id', 'gene_id', 'cell_line')
    .agg(
        F.avg('avg_dist').alias('avg_dist'),
        F.min('avg_dist').alias('min_dist'),
        # F.expr("""
        # aggregate(
        #     slice(sort_array(collect_list(avg_dist)), 1, 3),
        #     cast(0 as float),
        #     (acc, x) -> acc + x
        # ) / size(slice(sort_array(collect_list(avg_dist)), 1, 3))
        # """).alias('min_dist'),
        F.max('avg_dist').alias('max_dist'),
    )
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
results_by_gene.repartition(1).write.mode('overwrite').parquet("s3a://database/closest_enh_distance_by_gene")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…